# 📝 LangChain 에이전트와 도구 과제 LV2(응용) — RAG 체인·SQL·외부 API

> LV1 에서 익힌 도구·궤적을 **조합**합니다. RAG 를 체인으로 조립하고, 데이터베이스와 외부 API 를 도구로 연결합니다.

## 풀이 방법
1. 맨 위 **준비 셀들**(제공 코드)을 위에서부터 실행하세요. `.env` 에 본인 **`OPENAI_API_KEY`** 가 필요합니다. **데이터베이스는 준비물이 없습니다** — SQL 단원에서 배운 sqlite 를 쓰므로 접속 정보가 필요 없습니다.
2. 모델이 만든 답과 도구 선택은 **실행할 때마다 달라집니다** — 채점은 **타입·구조**로 합니다.

화이팅!

아래 준비 셀들을 먼저 실행하세요(임베딩 모델 로딩에 잠시 걸립니다).

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 서점 FAQ 문서를 LangChain 부품으로 색인합니다(임베딩 모델을 내려받느라 처음 한 번은 잠시 걸립니다).
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

faq_df = pd.read_csv('data/bookstore_faq.csv')

# 표의 한 행 = Document 하나. page_content 는 검색 대상 본문, metadata 는 함께 붙일 꼬리표입니다.
faq_docs = [Document(page_content=row.text, metadata={'id': row.id, 'title': row.title})
            for row in faq_df.itertuples()]

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 임베딩 단원에서 쓴 그 한국어 문장 임베딩 모델

# ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 문서가 중복되지 않습니다.
faq_store = Chroma.from_documents(faq_docs, embeddings, collection_name='bookstore_faq',
                                  ids=faq_df['id'].tolist())
faq_retriever = faq_store.as_retriever(search_kwargs={'k': 2})   # 질문마다 가장 가까운 2개를 돌려주는 검색기

print('색인 완료 — 문서 수:', len(faq_docs))

In [ ]:
# [제공 코드] 에이전트 공통 준비
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool          # 3번에서 도구를 직접 만들 때 씁니다

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 1. RAG 체인 직접 조립하기
**배경**: 교안에서 본 것처럼, 도구도 에이전트도 없이 **검색 → 프롬프트 → 모델 → 문자열**을 한 줄로 잇는 **체인**을 직접 조립합니다. 맨 위 준비 셀이 만들어 둔 **`faq_retriever`** 를 씁니다.

**요구사항**: 두 가지를 만드세요.

**(1) 기본 RAG 체인 `rag_chain`**

- `ChatPromptTemplate` 로 프롬프트를 만드세요. **`{context}`** 와 **`{question}`** 두 자리를 두고, *주어진 자료에 있는 내용만으로 한국어로 답하라*는 지시를 담습니다.
- 체인은 **검색 결과를 글로 합친 것**을 `context` 에, **질문 그대로**를 `question` 에 채운 뒤 프롬프트 → 모델 → 출력파서를 차례로 잇습니다. 질문을 그대로 흘려보내는 데에는 **`RunnablePassthrough`**, 최종 결과를 문자열로 받는 데에는 **`StrOutputParser`** 를 씁니다.
- 검색 결과를 한 덩어리 글로 합치는 **`format_docs`** 는 아래 제공 셀에 있습니다.
- 질문 **"전자책은 몇 대의 기기에서 볼 수 있나요?"** 로 `invoke` 한 결과를 변수 **`answer1`** 에 담으세요(문자열입니다).

**(2) 근거를 함께 돌려주는 체인 `chain_with_sources`**

- 실무에서 **출처 없는 RAG 답**은 쓰기 어렵습니다. **`RunnableParallel`** 로 두 갈래를 묶어 답과 근거를 함께 받으세요 — **`'answer'`** 에는 (1)의 체인을, **`'sources'`** 에는 검색기를 그대로 둡니다.
- 같은 질문 **"전자책은 몇 대의 기기에서 볼 수 있나요?"** 로 `invoke` 한 결과를 변수 **`result1`** 에 담으세요. `result1['answer']` 는 문자열, `result1['sources']` 는 **`Document` 목록**이 됩니다.

**예시**: `answer1` 은 전자책 기기 수를 설명하는 한국어 문장입니다(**문장은 실행할 때마다 달라지므로** 채점은 **타입·구조**로만 합니다). `result1['sources']` 의 각 `Document` 에는 `metadata['title']` 이 붙어 있어 어느 FAQ 에서 왔는지 알 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 딕셔너리로 두 자리를 채우고, 파이프로 프롬프트·모델·파서를 잇는다.

세부구현:
1. ChatPromptTemplate 로 자료와 질문 자리를 가진 프롬프트를 만든다.
   1-1. 자료에 있는 내용만으로 답하라는 지시를 넣는다.
2. context 자리에는 검색기와 format_docs 를 이어 붙인 것을, question 자리에는
   입력을 그대로 흘려보내는 부품을 둔다.
3. 그 딕셔너리에 프롬프트·모델·문자열 출력파서를 차례로 이어 체인을 만든다.
4. 답과 근거를 함께 받는 체인은 두 갈래를 나란히 묶는 부품으로 만든다.
   4-1. 한 갈래는 앞에서 만든 체인, 다른 갈래는 검색기 그대로다.
5. 두 체인에 같은 질문 문자열을 넣어 결과를 각각 담는다.
```

</details>

In [ ]:
# [제공 코드] 검색 결과(Document 목록)를 프롬프트에 넣을 한 덩어리 글로 합칩니다 — 실행만 하세요.
def format_docs(docs):
    """검색된 Document 들을 제목과 함께 한 덩어리 글로 합친다."""
    return '\n\n'.join(f"[{d.metadata['title']}] {d.page_content}" for d in docs)


print('준비 완료 —', format_docs(faq_retriever.invoke('전자책'))[:40], '...')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 실호출이라 문장은 매번 다르다 — 타입과 구조만 본다
assert isinstance(answer1, str) and len(answer1.strip()) > 0
assert isinstance(result1, dict) and set(result1) == {'answer', 'sources'}, \
    "두 갈래의 이름은 'answer' 와 'sources' 입니다"
assert isinstance(result1['answer'], str) and len(result1['answer'].strip()) > 0
assert isinstance(result1['sources'], list) and len(result1['sources']) >= 1
assert all(isinstance(d, Document) for d in result1['sources']), 'sources 는 Document 목록입니다'
assert all('title' in d.metadata for d in result1['sources'])
# sources 가 정말 검색기에서 왔는지 — 같은 질문을 검색기에 직접 넣어 대조한다(검색은 결정적이다)
want1 = [d.metadata['id'] for d in faq_retriever.invoke('전자책은 몇 대의 기기에서 볼 수 있나요?')]
assert [d.metadata['id'] for d in result1['sources']] == want1, \
    'sources 갈래에 검색기를 그대로 두세요'
print('✅ 통과!')

## 2. Text-to-SQL — 주문 건수 조회, 그리고 답을 스키마로
**배경**: 자연어 질문을 SQL 로 바꿔 답하게 합니다. **SELECT 전용** 도구와 **스키마 안내 프롬프트**를 함께 씁니다. 그다음, 같은 질문의 답을 **문장이 아니라 데이터**로 받아 봅니다.

**요구사항**: 두 가지를 만드세요.

**(1) 궤적 읽기**

- 아래 준비 셀들(`run_select` 도구·`SCHEMA_PROMPT`·데이터베이스·`OrderAnswer` 스키마)을 먼저 실행하세요.
- `create_agent(model, [run_select], system_prompt=SCHEMA_PROMPT)` 로 에이전트를 만들고, 질문 **"c1 고객이 주문한 건수는 모두 몇 건인가요?"** 로 `invoke` 한 결과를 변수 **`res2`** 에 담으세요.
- 궤적에서 **ToolMessage** 만 골라 변수 **`tool_msgs2`** 에 담으세요.

**(2) 도구도 쓰고, 답도 스키마로 — `response_format`**

- (1)의 최종 답은 **자유 문장**이라 그대로는 표에 넣지 못합니다. 교안 01 3절에서 본 것처럼 `create_agent` 에 **`response_format=ProviderStrategy(OrderAnswer, strict=True)`** 를 더해 에이전트를 하나 더 만드세요(임포트는 `from langchain.agents.structured_output import ProviderStrategy`).
- **같은 질문** "c1 고객이 주문한 건수는 모두 몇 건인가요?" 로 `invoke` 한 결과를 변수 **`res2b`** 에 담고, 거기서 정형 결과를 꺼내 변수 **`report2`** 에 담으세요. 정형 결과는 결과 딕셔너리의 **`'structured_response'`** 열쇠에 들어 있습니다.

**예시**: c1 고객의 주문은 3건이라 (1)의 도구 결과 어딘가에 **'3'** 이 들어 있습니다(모델이 표를 먼저 둘러보느라 조회를 여러 번 할 수도 있습니다). (2)의 `report2` 는 `OrderAnswer` 객체이고, `report2.order_count` 는 **3**, `report2.sql` 에는 실행한 **`select`** 문이, `report2.answer` 에는 사용자에게 보여 줄 한국어 한 문장이 들어 있습니다. 문장 내용은 실행마다 다릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 스키마 프롬프트를 준 SQL 에이전트로 질문을 실행하고, 도구 결과를 읽는다.
- 답까지 정형화하려면 에이전트를 만들 때 인자를 하나 더 준다 - 도구 목록은 그대로다.

세부구현:
1. create_agent 에 [run_select] 와 system_prompt=SCHEMA_PROMPT 를 준다.
2. 질문을 그대로 invoke 해 res2 에 담는다.
3. ToolMessage 만 골라 tool_msgs2 에 담는다.
4. 같은 인자에 response_format 을 더해 두 번째 에이전트를 만들고 같은 질문을 invoke 한다.
   4-1. 결과 딕셔너리에 키가 하나 더 생겨 있다 - 궤적은 그대로 남는다.
5. 그 키에서 정형 결과를 꺼내 report2 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 데이터베이스 준비 — SQL 단원에서 배운 sqlite 를 그대로 씁니다(접속 정보가 필요 없습니다).
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('output') / 'bookstore.db'
DB_PATH.parent.mkdir(exist_ok=True)
DB_PATH.unlink(missing_ok=True)          # 여러 번 실행해도 늘 같은 초기 상태에서 시작합니다

_conn = sqlite3.connect(DB_PATH, isolation_level=None)   # isolation_level=None : 실행 즉시 저장
_conn.execute('pragma foreign_keys = on')                # 외래키 검사를 켭니다(기본값은 꺼짐)
_conn.executescript(Path('data/setup_day19.sql').read_text(encoding='utf-8'))
_conn.execute('pragma foreign_keys = on')                # executescript 뒤에 한 번 더 켭니다

# 에이전트에게 줄 연결은 따로 만들고 '읽기 전용'으로 엽니다 — 모델이 무슨 SQL 을 만들든 쓰기가 막힙니다.
#  check_same_thread=False : 에이전트는 도구를 별도 스레드에서 실행하므로 이 옵션이 없으면 도구가 전부 실패합니다.
_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True,
                           isolation_level=None, check_same_thread=False)


def run_query(sql):
    """SELECT 결과를 DataFrame 으로 돌려준다(사람이 눈으로 확인할 때 쓴다)."""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print('데이터베이스 준비 완료 —', DB_PATH)

In [ ]:
# [제공 코드] 데이터베이스 조회 도구 — 에이전트가 이 도구로 SQL 을 실행합니다.
#  가드가 두 겹입니다: (1) 여기서 문장을 검사하고 (2) 연결 자체가 읽기 전용입니다.
from langchain_core.tools import tool


@tool
def run_select(sql: str) -> str:
    """읽기 전용 SQL(SELECT) 한 문장을 실행하고 결과를 문자열로 돌려준다. SELECT 한 문장이 아니면 거부한다."""
    stmt = sql.strip().rstrip(';')          # 끝의 세미콜론 하나는 흔한 표기라 허용한다
    # 세미콜론이 남아 있으면 문장이 둘 이상이라는 뜻 — 'select 1; delete ...' 를 막는다.
    if not stmt.lower().startswith('select') or ';' in stmt:
        return '거부: 이 도구는 SELECT 조회 한 문장만 실행할 수 있습니다.'
    try:
        return str(_ro_conn.execute(stmt).fetchall())   # 검사한 문장을 그대로 실행한다
    except Exception as e:
        return f'에러: {e}'                             # 에러도 문자열로 — 모델이 읽고 고쳐 다시 시도한다


print('SQL 도구 준비:', run_select.name)

In [ ]:
# [제공 코드] Text-to-SQL 용 스키마 안내 프롬프트
SCHEMA_PROMPT = (
    '너는 온라인 서점 데이터베이스 조회를 돕는 도우미다. run_select 도구로 SELECT 문만 실행해 답하라. 표 스키마는 다음과 같다. bs_customer(customer_id, name, grade, city): 고객. bs_book(book_id, title, author, genre, price, stock): 도서. bs_order(order_id, customer_id, book_id, quantity, order_date): 주문. 조회 결과를 바탕으로 한국어로 간단히 답하라.'
)

In [ ]:
# [제공 코드] 2번 (2)에서 쓸 결과 스키마 — 이 셀은 실행만 하세요.
from pydantic import BaseModel, Field


class OrderAnswer(BaseModel):
    """주문 건수 문의 한 건을 처리한 결과."""

    sql: str = Field(description="조회에 사용한 SELECT 문")
    # 이 칸은 모델이 지어내는 값이 아니라 '도구가 알려 준 값'이다.
    order_count: int = Field(description="도구가 알려 준 주문 건수")
    answer: str = Field(description="사용자에게 보여 줄 한국어 한 문장")


print("스키마 준비 완료")

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert tool_msgs2 == [m for m in res2['messages'] if isinstance(m, ToolMessage)]
assert res2['messages'][0].text.strip() == 'c1 고객이 주문한 건수는 모두 몇 건인가요?', '지문의 질문을 그대로 넣어 실행하세요'
assert len(tool_msgs2) >= 1, 'SQL 도구가 한 번도 불리지 않았습니다'
assert any('3' in str(m.content) for m in tool_msgs2)   # c1 의 주문 건수 3
assert isinstance(res2['messages'][-1].text, str)

# (2) 정형 결과 — 값이 아니라 '모양'을 스키마가 보장한다
assert report2 is res2b['structured_response'], 'report2 는 res2b 에서 꺼낸 정형 결과여야 합니다'
assert isinstance(report2, OrderAnswer)
# response_format 을 줘도 도구는 그대로 불린다 — 궤적이 남아 있는지 본다
assert any(isinstance(m, ToolMessage) for m in res2b['messages']), \
    'response_format 을 줘도 도구 궤적은 남습니다 — 도구가 불리지 않았습니다'
assert report2.order_count == 3, 'order_count 는 도구가 알려 준 주문 건수(3)여야 합니다'
assert 'select' in report2.sql.lower(), 'sql 칸에는 조회에 쓴 SELECT 문이 들어갑니다'
assert isinstance(report2.answer, str) and report2.answer.strip()
print('✅ 통과!')

## 3. 외부 API 를 도구로 — 오늘의 엔화 환율
**배경**: 지금까지 만든 도구는 전부 **노트북 안의 값**으로 답했습니다. 오늘의 값이 필요하면 **인터넷에 직접 물어봐야** 합니다. 이번엔 달러가 아니라 **엔화**로, 그리고 **금액을 인자로 받는** 도구를 직접 만듭니다.

`https://api.frankfurter.dev/v1/latest` 에 `amount`·`from`·`to` 를 붙여 요청하면 환산 결과가 옵니다 (키가 필요 없는 공개 API). 예를 들어 `{'amount': 100, 'from': 'JPY', 'to': 'KRW'}` 로 요청하면 응답 JSON 의 `rates['KRW']` 에 **100엔에 해당하는 원화**가 들어 있습니다.

> **인터넷은 언제든 끊깁니다.** 교안 01 4절의 **도구 설계 원칙 3 — 실패는 예외가 아니라 문자열로 돌려준다** 를 이 도구에 적용하세요. 예외를 밖으로 던지면 에이전트가 그 자리에서 멈추지만, 문자열이면 모델이 그것을 읽고 사용자에게 안내하거나 다시 시도할 수 있습니다.

**요구사항**:
- `@tool` 을 붙인 함수 **`jpy_krw_rate(amount_jpy: int) -> str`** 를 정의하세요. docstring 을 한국어로 적습니다.
- `import requests` 한 뒤 함수 안에서 **`requests.get(...)`** 으로 위 주소에 요청하세요. `params={'amount': amount_jpy, 'from': 'JPY', 'to': 'KRW'}` 와 **`timeout=10`** 을 줍니다.
- 요청과 `raise_for_status()`·`.json()` 을 **`try` 안에** 두고, **`except requests.RequestException`** 으로 실패를 잡으세요. 잡았을 때는 **예외를 다시 던지지 말고**, 왜 실패했는지 담은 **문자열**을 돌려줍니다.
- 성공했을 때 반환 문자열은 **`f'{amount_jpy}엔 = {round(원화)}원'`** 형식입니다(원화는 `rates['KRW']`, **반올림해 정수**로).
- `create_agent(model, [jpy_krw_rate])` 로 에이전트를 만들고, 질문 **"300엔은 우리 돈으로 얼마야?"** 로 `invoke` 한 결과를 변수 **`res3`** 에 담은 뒤, 불린 도구 이름 목록을 **`used3`** 에 담으세요.

**예시**: `jpy_krw_rate.invoke({'amount_jpy': 100})` → `'100엔 = 921원'` 같은 문자열(**환율은 날마다 바뀌므로 숫자는 오늘 값**입니다). 요청이 실패하면 `'환율을 가져오지 못했습니다: ...'` 처럼 **사정을 알리는 문자열**이 나옵니다. `used3` 에는 `'jpy_krw_rate'` 가 들어 있습니다.

> **채점 안내**: 자가채점은 성공 경로뿐 아니라 **실패 경로도** 확인합니다 — `requests.get` 이 실패하도록 잠깐 바꿔 놓고 도구를 불러, 예외가 밖으로 나오지 않고 **문자열**이 돌아오는지 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 데이터 수집 단원에서 배운 requests 요청과 JSON 읽기를 @tool 함수 안에 넣되, 끊길 때에 대비해 try 로 감싼다.

세부구현:
1. requests 를 임포트하고 @tool 과 타입힌트·docstring 을 단다.
2. try 안에서 params 에 amount·from·to 를 넣고 timeout 을 준 뒤,
   raise_for_status 로 실패를 확인하고 응답 JSON 을 읽는다.
3. except 로 requests 의 요청 계열 예외를 잡고, 다시 던지지 말고
   실패를 알리는 문자열을 반환한다(도구 설계 원칙 3).
4. 성공하면 응답에서 원화 값을 꺼내 반올림해 지정된 형식의 문자열로 만든다.
5. 만든 도구 하나로 에이전트를 만들고 질문을 invoke 한 뒤 ToolMessage 의 name 을 모은다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import requests as rq

# 채점도 같은 API 에 직접 물어본다 — 값을 손으로 박아 두면 오늘 환율과 어긋나 걸린다
#  채점 자신이 망을 타므로, 여기서 끊기면 '내 코드가 틀린 건가' 하고 헤매게 된다.
#  그래서 채점의 요청도 감싸서, 망 문제일 때는 그렇다고 말해 준다.
try:
    ref3 = rq.get('https://api.frankfurter.dev/v1/latest',
                  params={'amount': 100, 'from': 'JPY', 'to': 'KRW'}, timeout=10).json()
except Exception as e:
    raise AssertionError(
        f'채점이 환율 API 에 닿지 못했습니다({type(e).__name__}) — 인터넷 연결을 확인하세요. '
        '여러분의 코드 문제가 아닙니다.') from None

got3 = jpy_krw_rate.invoke({'amount_jpy': 100})
assert isinstance(got3, str)
assert '100엔' in got3
assert f"{round(ref3['rates']['KRW'])}원" in got3, '오늘 환율로 환산한 값이 보이지 않습니다'

# 실패 경로도 본다 — 요청이 실패하도록 requests.get 을 잠깐 바꿔 놓고 도구를 부른다
saved_get = rq.get


def failing_get(*args, **kwargs):
    """채점용 모의 상황 — 인터넷이 끊긴 것처럼 요청을 실패시킨다."""
    raise rq.RequestException('연결 실패(채점용 모의 상황)')


rq.get = failing_get
try:
    fail3 = jpy_krw_rate.invoke({'amount_jpy': 100})
finally:
    rq.get = saved_get                     # 무슨 일이 있어도 원래대로 되돌린다
assert isinstance(fail3, str), '실패해도 예외를 던지지 말고 문자열을 돌려주세요(도구 설계 원칙 3)'

assert used3 == [m.name for m in res3['messages'] if isinstance(m, ToolMessage)]
assert res3['messages'][0].text.strip() == '300엔은 우리 돈으로 얼마야?', '지문의 질문을 그대로 넣어 실행하세요'
assert 'jpy_krw_rate' in used3, '에이전트가 도구를 부르지 않았습니다'
print('✅ 통과!')

---
수고했어요! RAG 를 **체인으로 직접 조립**해 근거까지 함께 받았습니다. 데이터베이스를 도구로 붙여 **자연어 질문을 SQL 로** 바꾸고 그 답까지 **스키마에 담아** 받았으며, 마지막으로 외부 API 를 도구로 감싸며 **실패를 문자열로 돌려주는** 습관까지 손에 익혔습니다. LV3 에서는 이것들을 묶어 **리뷰 인텔리전스 파이프라인**을 만듭니다.